# 01 · Data Exploration
**Project 18 — Decision Network for Intelligent Recommendation Systems**  
Dataset: MovieLens 100K

In this notebook we:
1. Download and load the MovieLens 100K dataset
2. Explore the rating distribution, user demographics, and movie metadata
3. Visualise key patterns that motivate our Decision Network design


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

try:
    from src.data_loader import load_all, train_test_split_temporal
except ImportError:
    from data_loader import load_all, train_test_split_temporal

# ── Style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0f1525',
    'axes.facecolor'   : '#0f1525',
    'axes.edgecolor'   : '#2d3748',
    'axes.labelcolor'  : '#a0aec0',
    'xtick.color'      : '#718096',
    'ytick.color'      : '#718096',
    'text.color'       : '#e2e8f0',
    'grid.color'       : '#1a2035',
    'grid.linestyle'   : '--',
    'font.family'      : 'DejaVu Sans',
})
PALETTE = ['#8b5cf6', '#06b6d4', '#10b981', '#f59e0b', '#f43f5e']
sns.set_palette(PALETTE)

print('Libraries loaded ✓')

In [ ]:
# ── Load all data ─────────────────────────────────────────────
ratings, users, items, merged = load_all()
train_r, test_r = train_test_split_temporal(ratings)
print(f'Train: {len(train_r):,} | Test: {len(test_r):,}')
ratings.head()

## 1.1  Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Rating Distributions', fontsize=14, fontweight='bold', color='#e2e8f0', y=1.02)

# Overall distribution
ax = axes[0]
counts = ratings['rating'].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=PALETTE[0], width=0.6, alpha=0.85, edgecolor='#8b5cf6')
ax.set_title('Overall Rating Distribution', color='#e2e8f0')
ax.set_xlabel('Rating (1–5)'); ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200, f'{val:,}',
            ha='center', va='bottom', fontsize=8, color='#a78bfa')

# Ratings per user
ax = axes[1]
user_counts = ratings.groupby('user_id')['rating'].count()
ax.hist(user_counts, bins=40, color=PALETTE[1], edgecolor='#0f1525', alpha=0.85)
ax.axvline(user_counts.median(), color='#f43f5e', linewidth=2, linestyle='--',
           label=f'Median={user_counts.median():.0f}')
ax.set_title('Ratings per User', color='#e2e8f0')
ax.set_xlabel('Number of Ratings'); ax.set_ylabel('Users')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Ratings per movie
ax = axes[2]
item_counts = ratings.groupby('item_id')['rating'].count()
ax.hist(item_counts, bins=40, color=PALETTE[2], edgecolor='#0f1525', alpha=0.85)
ax.axvline(item_counts.median(), color='#f43f5e', linewidth=2, linestyle='--',
           label=f'Median={item_counts.median():.0f}')
ax.set_title('Ratings per Movie', color='#e2e8f0')
ax.set_xlabel('Number of Ratings'); ax.set_ylabel('Movies')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_rating_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Sparsity: {1 - len(ratings) / (ratings.user_id.nunique() * ratings.item_id.nunique()):.3%}')

## 1.2  User Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('User Demographics (MovieLens 100K)', fontsize=14, fontweight='bold', color='#e2e8f0')

# Gender
ax = axes[0]
gender_counts = users['gender'].value_counts()
ax.pie(gender_counts, labels=['Male', 'Female'], colors=[PALETTE[0], PALETTE[4]],
       autopct='%1.1f%%', startangle=90, textprops={'color': '#e2e8f0', 'fontsize': 11})
ax.set_title('Gender Split', color='#e2e8f0')

# Age distribution
ax = axes[1]
ax.hist(users['age'], bins=20, color=PALETTE[1], edgecolor='#0f1525', alpha=0.85)
ax.axvline(users['age'].median(), color='#f43f5e', linewidth=2, linestyle='--',
           label=f"Median={users['age'].median():.0f}")
ax.set_title('Age Distribution', color='#e2e8f0')
ax.set_xlabel('Age'); ax.set_ylabel('Count'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Top occupations
ax = axes[2]
top_occ = users['occupation'].value_counts().head(10)
ax.barh(top_occ.index, top_occ.values, color=PALETTE[2], alpha=0.85, edgecolor='#10b981')
ax.set_title('Top 10 Occupations', color='#e2e8f0')
ax.set_xlabel('Count'); ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../data/fig_user_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3  Genre Analysis

In [ ]:
try:
    from src.data_loader import GENRE_COLUMNS
except ImportError:
    from data_loader import GENRE_COLUMNS

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Genre frequency
ax = axes[0]
genre_totals = items[GENRE_COLUMNS].sum().sort_values(ascending=True)
colors_bar = plt.cm.plasma(np.linspace(0.2, 0.9, len(genre_totals)))
ax.barh(genre_totals.index, genre_totals.values, color=colors_bar, edgecolor='none')
ax.set_title('Movies per Genre', fontsize=12, color='#e2e8f0')
ax.set_xlabel('Number of Movies'); ax.grid(axis='x', alpha=0.3)

# Avg rating by genre
ax = axes[1]
genre_ratings = (
    merged.groupby('primary_genre')['rating']
    .agg(['mean', 'count'])
    .query('count >= 50')
    .sort_values('mean', ascending=True)
)
ax.barh(genre_ratings.index, genre_ratings['mean'],
        color=PALETTE[0], alpha=0.8, edgecolor='none')
ax.axvline(genre_ratings['mean'].mean(), color='#f43f5e', linewidth=2,
           linestyle='--', label='Overall mean')
ax.set_title('Average Rating by Genre (≥50 ratings)', fontsize=12, color='#e2e8f0')
ax.set_xlabel('Average Rating'); ax.set_xlim(3.0, 4.2)
ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_genre_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4  Temporal Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Ratings over time
ax = axes[0]
ratings_by_month = ratings.groupby(ratings['datetime'].dt.to_period('M')).size()
ax.fill_between(range(len(ratings_by_month)), ratings_by_month.values,
                alpha=0.4, color=PALETTE[0])
ax.plot(range(len(ratings_by_month)), ratings_by_month.values,
        color=PALETTE[0], linewidth=2)
ax.set_title('Ratings Volume Over Time', color='#e2e8f0')
ax.set_xlabel('Month (Sep 1997 – Apr 1998)'); ax.set_ylabel('Ratings')
ax.grid(alpha=0.3)

# Day-of-week pattern
ax = axes[1]
dow_avg = merged.groupby('weekday')['rating'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax.bar(days, dow_avg.values, color=PALETTE[2], alpha=0.85, edgecolor='none')
ax.set_title('Average Rating by Day of Week', color='#e2e8f0')
ax.set_ylabel('Average Rating')
ax.set_ylim(3.3, 3.7); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Notebook 01 complete. Proceed to 02_feature_engineering.ipynb')